# Emoji Sentiment

**Question:** Are popular emojis generally associated with positive or negative sentiments?

The file `emoji-sentiment.csv` provides data on the sentiment associated with various emojis.
Researchers examined 1.6 million tweets across 13 European languages. Each tweet was labeled by annotators as positive (+1), negative (-1) or neutral (0). Only a small share (about 4%) of those tweets contained emojis, and only emojis with at least 5 occurrences are included.

Columns:
- `Occurrences [5...max]`: number of times the emoji appears in the dataset
- `Position [0...1]`: average position of the emoji in a tweet (0 = start, 1 = end)
- `Neg / Neut / Pos [0...1]`: share of tweets containing the emoji that are negative / neutral / positive

**Key analysis choice:** every row is one *emoji type*. A rare emoji (5 uses) and 😂 (14,000+ uses) are one row each, so a plain average over rows describes emoji *types*, not how emojis are *used*. This notebook reports both views, weighting by `occurrences` for the "usage" view.

## 1. Setup

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Colours used consistently across all charts
COL_POS, COL_NEU, COL_NEG = "#2ec4b6", "#9aa5b1", "#e71d36"

# Net sentiment beyond +/- this value counts as clearly positive / negative.
# Emojis in between are "mixed/neutral". Used for the class-based analysis below.
CLASS_THRESHOLD = 0.10

## 2. Load the data

Works in Google Colab and locally. If a file named `emoji-sentiment*.csv` is already in the working directory it is used (the newest one if there are several, e.g. `emoji-sentiment (1).csv`). Otherwise, in Colab, an upload dialog opens.

In [ ]:
def find_csv():
    matches = glob.glob("emoji-sentiment*.csv")
    if matches:
        return max(matches, key=os.path.getmtime)
    try:
        from google.colab import files  # only available in Colab
    except ImportError:
        raise FileNotFoundError(
            "Place 'emoji-sentiment.csv' in the working directory and re-run this cell."
        )
    uploaded = files.upload()
    return next(iter(uploaded))  # name of the file that was actually uploaded


csv_path = find_csv()
print(f"Loading: {csv_path}")
raw = pd.read_csv(csv_path)
raw.head(3)

## 3. Sanity checks on the raw data

In [ ]:
print(f"Shape: {raw.shape}")
print("\nMissing values per column:")
print(raw.isna().sum().to_string())

print(f"\nDuplicate emoji characters: {raw['Char'].duplicated().sum()}")
print(f"Duplicate unicode codepoints: {raw['Unicode codepoint'].duplicated().sum()}")

# The three sentiment shares should add up to ~1 for every emoji
share_sum = raw[["Neg [0...1]", "Neut [0...1]", "Pos [0...1]"]].sum(axis=1)
print(f"\nNeg+Neut+Pos: min={share_sum.min():.3f}, max={share_sum.max():.3f}")
print(f"Position range: {raw['Position [0...1]'].min():.3f} to {raw['Position [0...1]'].max():.3f}")
print(f"Occurrences: min={raw['Occurrences [5...max]'].min()}, max={raw['Occurrences [5...max]'].max()}")

## 4. Data cleaning

Columns are renamed explicitly (clearer and safer than regex clean-up, which leaves trailing underscores such as `position_`). Columns that are empty or redundant for this analysis are dropped: `Image [twemoji]` duplicates `Char`, and `Sentiment bar (c.i. 95%)` is empty.

In [ ]:
df = raw.rename(columns={
    "Char": "char",
    "Unicode codepoint": "unicode_codepoint",
    "Occurrences [5...max]": "occurrences",
    "Position [0...1]": "position",
    "Neg [0...1]": "neg",
    "Neut [0...1]": "neu",
    "Pos [0...1]": "pos",
    "Unicode name": "unicode_name",
    "Unicode block": "unicode_block",
})

# No errors="ignore": if a column name is wrong we want to know
df = df.drop(columns=["Image [twemoji]", "Sentiment bar (c.i. 95%)"])

# Readable names for charts and tables
df["unicode_name"] = df["unicode_name"].str.title()

print(df.columns.tolist())
df.head(3)

## 5. New variables

- `sentiment` = `pos` - `neg` (ranges from -1 to +1)
- `positive_flag` = `True` if `sentiment > 0` (the definition from the project brief)
- `sentiment_class` = `positive` if `sentiment > CLASS_THRESHOLD`, `negative` if `sentiment < -CLASS_THRESHOLD`, otherwise `mixed/neutral`. This three-way split is used when comparing positive and *negative* emojis, so that near-zero emojis don't blur the comparison.

In [ ]:
df["sentiment"] = df["pos"] - df["neg"]
df["positive_flag"] = df["sentiment"] > 0

df["sentiment_class"] = np.select(
    [df["sentiment"] > CLASS_THRESHOLD, df["sentiment"] < -CLASS_THRESHOLD],
    ["positive", "negative"],
    default="mixed/neutral",
)
df["sentiment_class"] = pd.Categorical(
    df["sentiment_class"], categories=["positive", "mixed/neutral", "negative"], ordered=True
)

df[["char", "unicode_name", "occurrences", "sentiment", "positive_flag", "sentiment_class"]].head()

## 6. Analysis

### Q1. What percentage of emojis have a positive sentiment?

Two answers, because "emojis" can mean *types* (rows) or *uses* (weighted by occurrences).

In [ ]:
total_uses = df["occurrences"].sum()

pct_types_pos = df["positive_flag"].mean() * 100
pct_usage_pos = df.loc[df["positive_flag"], "occurrences"].sum() / total_uses * 100

print(f"Share of emoji TYPES with positive sentiment : {pct_types_pos:.2f}%  ({df['positive_flag'].sum()} of {len(df)})")
print(f"Share of emoji USES  with positive sentiment : {pct_usage_pos:.2f}%")

**Sensitivity check:** does the answer depend on where the "positive" line is drawn? A threshold of 0 counts an emoji with net sentiment of +0.001 as positive.

In [ ]:
rows = []
for t in [0, 0.05, 0.10, 0.20, 0.30]:
    flag = df["sentiment"] > t
    rows.append({
        "threshold": t,
        "% of types positive": round(flag.mean() * 100, 1),
        "% of usage positive": round(df.loc[flag, "occurrences"].sum() / total_uses * 100, 1),
    })
pd.DataFrame(rows).set_index("threshold")

### Q2. What percentage of the 20 most popular emojis are positive?

In [ ]:
top_20 = df.nlargest(20, "occurrences")
pct_top20_pos = top_20["positive_flag"].mean() * 100
print(f"Top 20 most popular emojis that are positive: {pct_top20_pos:.0f}%")

print("\nThe exceptions (not positive):")
display(
    top_20.loc[~top_20["positive_flag"], ["char", "unicode_name", "occurrences", "pos", "neg", "sentiment"]]
)

print("The most popular emoji and how mild its positivity is:")
display(top_20.head(1)[["char", "unicode_name", "occurrences", "pos", "neu", "neg", "sentiment"]])

### Q3 & Q4. Most positive and most negative emojis (more than 500 mentions)

In [ ]:
popular = df[df["occurrences"] > 500]
print(f"{len(popular)} emojis have more than 500 occurrences.\n")

cols = ["char", "unicode_name", "occurrences", "sentiment"]
print("Most positive:")
display(popular.nlargest(5, "sentiment")[cols].reset_index(drop=True))
print("Most negative:")
display(popular.nsmallest(5, "sentiment")[cols].reset_index(drop=True))

best, worst = popular.nlargest(1, "sentiment").iloc[0], popular.nsmallest(1, "sentiment").iloc[0]
print(f"Q3 answer: {best['char']} ({best['unicode_name']}), sentiment {best['sentiment']:+.3f}")
print(f"Q4 answer: {worst['char']} ({worst['unicode_name']}), sentiment {worst['sentiment']:+.3f}")

### Q5. Where in tweets are most emojis located?

`position` is itself an average per emoji, so the overall figure is an occurrence-weighted mean of those averages (the plain mean over rows is shown for comparison).

In [ ]:
mean_pos_weighted = np.average(df["position"], weights=df["occurrences"])
mean_pos_plain = df["position"].mean()

print(f"Average position, weighted by usage : {mean_pos_weighted:.3f}")
print(f"Average position, plain mean of rows: {mean_pos_plain:.3f}")
print("(0 = start of tweet, 1 = end of tweet)")

### Q6. Do positive and negative emojis sit in different places within a tweet?

Compares clearly **positive** vs clearly **negative** emojis (using `CLASS_THRESHOLD`), not positive vs "everything else". Because the gap may be small, a bootstrap confidence interval (resampling emoji types, keeping usage weights) shows whether it is distinguishable from zero.

In [ ]:
pos_df = df[df["sentiment_class"] == "positive"]
neg_df = df[df["sentiment_class"] == "negative"]

def wmean(d):
    return np.average(d["position"], weights=d["occurrences"])

print(f"Positive emojis (n={len(pos_df)}): weighted mean position = {wmean(pos_df):.3f}")
print(f"Negative emojis (n={len(neg_df)}): weighted mean position = {wmean(neg_df):.3f}")
observed_diff = wmean(pos_df) - wmean(neg_df)
print(f"Difference (positive - negative)            = {observed_diff:+.3f}")

# Bootstrap the difference
rng = np.random.default_rng(42)
diffs = np.empty(5000)
for i in range(len(diffs)):
    diffs[i] = wmean(pos_df.sample(len(pos_df), replace=True, random_state=rng)) - \
               wmean(neg_df.sample(len(neg_df), replace=True, random_state=rng))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"95% bootstrap CI for the difference         = [{lo:+.3f}, {hi:+.3f}]")
print("-> CI includes 0: no clear difference in placement." if lo <= 0 <= hi
      else "-> CI excludes 0: placement differs between positive and negative emojis.")

## 7. Charts

Notes: emoji glyphs are not drawn inside chart text because Matplotlib's default font cannot render most of them (they show as blank boxes). Emojis are identified by name instead, and the glyphs appear in the tables above.

### Chart 1. Share of emoji types vs. share of emoji usage, by sentiment class

In [ ]:
order = ["positive", "mixed/neutral", "negative"]
types_share = df["sentiment_class"].value_counts(normalize=True).reindex(order) * 100
usage_share = df.groupby("sentiment_class", observed=False)["occurrences"].sum().reindex(order) / total_uses * 100

x = np.arange(len(order))
width = 0.38
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - width / 2, types_share.values, width, label="Share of emoji types", color="#4a6fa5")
b2 = ax.bar(x + width / 2, usage_share.values, width, label="Share of emoji uses", color="#f4a259")

for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1, f"{b.get_height():.1f}%",
                ha="center", fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels([f"{o}" for o in order])
ax.set_ylabel("Percent")
ax.set_ylim(0, max(types_share.max(), usage_share.max()) + 12)
ax.set_title(f"Emoji sentiment classes (net sentiment beyond ±{CLASS_THRESHOLD})\n"
             "Emojis with 5+ uses in European-language tweets", fontsize=12, weight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("emoji_sentiment_distribution.png", dpi=300)
plt.show()

### Chart 2. Five most positive vs. five most negative emojis (more than 500 mentions)

In [ ]:
extremes = pd.concat([popular.nlargest(5, "sentiment"), popular.nsmallest(5, "sentiment")])
extremes = extremes.drop_duplicates("unicode_codepoint").sort_values("sentiment")

fig, ax = plt.subplots(figsize=(10, 6))
colors = [COL_NEG if s < 0 else COL_POS for s in extremes["sentiment"]]
bars = ax.barh(extremes["unicode_name"], extremes["sentiment"], color=colors, height=0.6)
ax.axvline(0, color="gray", linewidth=0.8, linestyle="--")

for bar, score, n in zip(bars, extremes["sentiment"], extremes["occurrences"]):
    ha = "right" if score < 0 else "left"
    offset = -0.01 if score < 0 else 0.01
    ax.text(score + offset, bar.get_y() + bar.get_height() / 2,
            f"{score:+.2f}  (n={n:,})", va="center", ha=ha, fontsize=10)

ax.set_xlim(-0.65, 1.2)
ax.set_xlabel("Net sentiment score  [(share positive) - (share negative)]")
ax.set_title("Most positive vs. most negative emojis (more than 500 mentions)",
             fontsize=13, weight="bold")
plt.tight_layout()
plt.savefig("top_extreme_emojis.png", dpi=300)
plt.show()

### Chart 3. Where emojis sit within tweets (weighted by usage)

Densities are weighted by `occurrences`, so frequently used emojis count more. Dashed lines mark each group's weighted mean. `cut=0` stops the curve extending beyond the 0-1 range.

In [ ]:
plot_df = df[df["sentiment_class"].isin(["positive", "negative"])].copy()
plot_df["sentiment_class"] = plot_df["sentiment_class"].astype(str)

fig, ax = plt.subplots(figsize=(10, 5))
sns.kdeplot(
    data=plot_df, x="position", weights="occurrences", hue="sentiment_class",
    hue_order=["positive", "negative"], palette={"positive": COL_POS, "negative": COL_NEG},
    common_norm=False, fill=True, alpha=0.35, linewidth=2, cut=0, ax=ax,
)

for cls, colr in [("positive", COL_POS), ("negative", COL_NEG)]:
    ax.axvline(wmean(plot_df[plot_df["sentiment_class"] == cls]), color=colr, linestyle="--", linewidth=1.5)

ax.set_xlim(0, 1)
ax.set_xlabel("Average position within tweet (0 = start, 1 = end)")
ax.set_ylabel("Density (usage-weighted)")
ax.set_title("Emoji placement in tweets: positive vs. negative emojis", fontsize=13, weight="bold")
ax.get_legend().set_title("Sentiment class")
plt.tight_layout()
plt.savefig("emoji_position_density.png", dpi=300)
plt.show()

## 8. Conclusions

In [ ]:
print(f"Of {len(df)} emoji types, {pct_types_pos:.1f}% are net-positive; weighted by how often they are used, {pct_usage_pos:.1f}%.")
print(f"{pct_top20_pos:.0f}% of the 20 most popular emojis are net-positive.")
print(f"Most positive (>500 uses): {best['unicode_name']} ({best['sentiment']:+.2f}); "
      f"most negative: {worst['unicode_name']} ({worst['sentiment']:+.2f}).")
print(f"Emojis sit late in tweets on average (weighted mean position {mean_pos_weighted:.2f}).")
print(f"Positive vs negative placement difference: {observed_diff:+.3f} (95% CI {lo:+.3f} to {hi:+.3f}).")

**Summary:** In the original run of this analysis, most emoji types (about 82%) and 18 of the 20 most popular emojis were net-positive, and emojis tended to appear toward the end of a tweet. Compare that with the figures printed above, including the usage-weighted share. Check the Q6 interval before claiming that positive and negative emojis are placed differently: if it includes 0, the data do not support a difference.

**Nuances worth keeping in mind**
- Being in the "top 20" does not guarantee positivity. Some very popular emojis (for example 😭, Loudly Crying Face) are net negative, because they are used for both joy and sadness.
- Even the most popular emoji, 😂, is only mildly positive and has a notable share of negative tweets.
- The share of "positive emojis" depends on the threshold chosen; see the sensitivity table.

**Limitations**
- Only emojis with at least 5 occurrences are included.
- The tweets are from 13 European languages, so the results may not generalise globally.
- Sentiment labels belong to the *tweet*, not to the emoji. A tweet's sentiment reflects all of its words, so emoji "sentiment" here means "sentiment of the tweets it appears in".
- Only a small fraction of the original 1.6M tweets contained emojis.